# Loading per Station POI data 
### Uses Geoapify API to query for per station data in a 400m radius (5 minute walk) around the station

### Features searched for include
- Bus station
- Subway station
- Streetcar station
- Train station
- Tourist locations
- Office buildings
- Parks
- Healthcare buildings
- Educational buildings
- Restaurants
- Commercial buildings (retail)

In [34]:
import os
import pandas as pd
from dotenv import load_dotenv
import requests

load_dotenv()

stations_df = pd.read_csv('bike_stations_missing_filled.csv', encoding='utf-8')
stations_df.info()
stations_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   station_id           1042 non-null   int64  
 1   name                 1042 non-null   object 
 2   lat                  1042 non-null   float64
 3   lon                  1042 non-null   float64
 4   capacity             1042 non-null   int64  
 5   address              983 non-null    object 
 6   is_charging_station  1042 non-null   bool   
 7   nearby_distance      984 non-null    float64
dtypes: bool(1), float64(3), int64(2), object(2)
memory usage: 58.1+ KB


,station_id,name,lat,lon,capacity,address,is_charging_station,nearby_distance
0,7000,Fort York Blvd / Capreol Ct,43.639832,-79.395954,47,Fort York Blvd / Capreol Ct,False,500.0
1,7001,Wellesley Station Green P,43.664964,-79.383550,23,Yonge / Wellesley,True,500.0
2,7002,St. George St / Bloor St W,43.667131,-79.399555,19,St. George St / Bloor St W,False,500.0
3,7003,Madison Ave / Bloor St W,43.667018,-79.402796,15,Madison Ave / Bloor St W,False,500.0
4,7005,King St W / York St,43.648001,-79.383177,26,King St W / York St,False,500.0


In [35]:
### Pull POI Data from Geoapify API ###
# Search Parameters
CATEGORY_MAPPING = {
    "bus": "public_transport.bus",
    "subway": "public_transport.subway",
    "streetcar": "public_transport.tram",
    "train": "public_transport.train",
    "all_public_transport": "public_transport",
    "tourism": "tourism",
    "office": "office",
    "park": "leisure.park",
    "healthcare": "healthcare",
    "education": "education",
    "food_drink": "catering",
    "commercial": "commercial"
}
CATEGORIES = ",".join(CATEGORY_MAPPING.values())
GEOAPIFY_KEY = os.environ.get('GEOAPIFY_KEY_2')
RADIUS_METERS = 400

# Search and Count Method
def fetch_poi_counts(row):
    """Fetches POI data and returns a dictionary of counts using simple column names."""
    lat, lon = row['lat'], row['lon']
    
    # [API Key and URL setup remains the same]
    url = (
        f"https://api.geoapify.com/v2/places?"
        f"categories={CATEGORIES}&"
        f"filter=circle:{lon},{lat},{RADIUS_METERS}&"
        f"limit=500&"
        f"apiKey={GEOAPIFY_KEY}"
    )
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() 
        data = response.json()
        
        # Initialize the dictionary keys using the simple column names
        poi_counts = {name: 0 for name in CATEGORY_MAPPING.keys()}
        
        for feature in data.get('features', []):
            cats = feature['properties'].get('categories', [])
            
            # Convert cats to a string once for efficient searching
            cats_string = str(cats) 
            
            # Loop through the mapping to find matches and use the simple name as the key
            for category, geoapify_cat in CATEGORY_MAPPING.items():
                
                # Check if the Geoapify category tag (the value) is present in the POI's category list
                if geoapify_cat in cats_string:
                    poi_counts[category] += 1
                
        return poi_counts
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for station {row['station_id']} ({row['name']}): {e}")
        # Return NaNs using the simple column names
        return {simple_name: float('nan') for simple_name in CATEGORY_MAPPING.keys()}

# Fetch POI data for all stations
all_poi_data = []

print("Starting API Calls...")
for index, row in stations_df.iterrows():
    counts = fetch_poi_counts(row)
    print(counts)
    
    # Add the station's ID to the counts dictionary for easy merging later
    counts['station_id'] = row['station_id']
    all_poi_data.append(counts)

print("Finished API calls. Merging data...")

# Convert the list of dictionaries into a new DataFrame
poi_df = pd.DataFrame(all_poi_data)

# Merge the new POI data back into your original DataFrame
stations_df = stations_df.merge(poi_df, on='station_id', how='left')

print("DataFrame updated successfully!")
print(stations_df[['station_id', 'name'] + list(poi_df.columns[1:])].head())

Starting API Calls...
{'bus': 0, 'subway': 0, 'streetcar': 11, 'train': 0, 'all_public_transport': 11, 'tourism': 20, 'office': 10, 'park': 43, 'healthcare': 15, 'education': 6, 'food_drink': 32, 'commercial': 56}
{'bus': 20, 'subway': 1, 'streetcar': 0, 'train': 0, 'all_public_transport': 21, 'tourism': 4, 'office': 11, 'park': 60, 'healthcare': 27, 'education': 8, 'food_drink': 126, 'commercial': 149}
{'bus': 14, 'subway': 1, 'streetcar': 2, 'train': 0, 'all_public_transport': 17, 'tourism': 3, 'office': 15, 'park': 19, 'healthcare': 7, 'education': 13, 'food_drink': 35, 'commercial': 31}
{'bus': 8, 'subway': 2, 'streetcar': 4, 'train': 0, 'all_public_transport': 14, 'tourism': 4, 'office': 10, 'park': 18, 'healthcare': 7, 'education': 13, 'food_drink': 40, 'commercial': 46}
{'bus': 8, 'subway': 2, 'streetcar': 10, 'train': 0, 'all_public_transport': 20, 'tourism': 12, 'office': 56, 'park': 41, 'healthcare': 25, 'education': 1, 'food_drink': 206, 'commercial': 149}
{'bus': 17, 'subwa

In [ ]:
print(stations_df.isnull().sum())
print(stations_df.info())
print(stations_df.head())

   station_id                          name        lat        lon  capacity  \
0        7000  Fort York  Blvd / Capreol Ct  43.639832 -79.395954        47   
1        7001     Wellesley Station Green P  43.664964 -79.383550        23   
2        7002    St. George St / Bloor St W  43.667131 -79.399555        19   
3        7003      Madison Ave / Bloor St W  43.667018 -79.402796        15   
4        7005           King St W / York St  43.648001 -79.383177        26   

                        address  is_charging_station  nearby_distance  bus  \
0  Fort York  Blvd / Capreol Ct                False            500.0    0   
1             Yonge / Wellesley                 True            500.0   20   
2    St. George St / Bloor St W                False            500.0   14   
3      Madison Ave / Bloor St W                False            500.0    8   
4           King St W / York St                False            500.0    8   

   subway  streetcar  train  all_public_transport  touri

### Only run the below code if there are rows of null values in the returned POIs

In [37]:
# stations_df_copy = stations_df.copy()

# POI_COLUMNS = [
#     "bus", "subway", "streetcar", "train", "all_public_transport",
#     "tourism", "office", "park", "healthcare", "education",
#     "food_drink", "commercial"
# ]

# is_any_poi_null = stations_df_copy[POI_COLUMNS].isnull().any(axis=1)

# failed_stations_df = stations_df_copy[is_any_poi_null]

# if failed_stations_df.empty:
#     print("✅ All POI data appears to be successfully loaded. No rows require retry.")
# else:
#     print(f"Detected {len(failed_stations_df)} stations requiring a retry...")
    
#     # --- 2. Execute Retry Loop ---
    
#     retried_poi_data = []
    
#     # Iterate over ONLY the failed rows
#     for index, row in failed_stations_df.iterrows():
#         print(f"  --> Retrying station {row['station_id']} ({row['name']})...")
        
#         # Call the existing fetching function (assumed to be defined elsewhere)
#         counts = fetch_poi_counts(row)
        
#         # Add the original station_id (which is needed for the final merge/update)
#         counts['station_id'] = row['station_id']
#         retried_poi_data.append(counts)

#         retried_df = pd.DataFrame(retried_poi_data)
    
#     # Set the index of the retried DataFrame to match the index of the main DataFrame.
#     # We reset the index on the main DataFrame first if it's not the station_id.
#     retried_df = retried_df.set_index('station_id')
    
#     # Ensure the main DataFrame is also indexed by station_id for clean update
#     stations_df_copy = stations_df_copy.set_index('station_id')

#     # --- 4. Update the Original DataFrame ---

#     # The update() method replaces non-NaN values in stations_df with values from retried_df 
#     # where the indices (station_id) match. This is the cleanest way to fill in missing data.
#     stations_df_copy.update(retried_df)

#     # Restore the original index if necessary
#     stations_df_copy = stations_df_copy.reset_index()
    
#     print("\n✅ Retry complete. Checking for remaining NaNs...")
    
#     # Re-check the count of NaNs in the POI columns
#     remaining_nulls = stations_df_copy[POI_COLUMNS].isnull().sum().sum()




In [40]:
stations_df.to_csv('final_bike_stations.csv', index=False, encoding='utf-8')